# Task 1. Dataset and Dependencies

Install and import all required dependencies

In [1]:
%pip install -q pgserver==0.1.4 psycopg2-binary sqlalchemy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 68.9 MB/s eta 0:00:00


In [2]:
import pgserver
import pandas as pd
from sqlalchemy import create_engine, text

# Dataset download and unpacking
from urllib.request import urlopen
from zipfile import ZipFile
from io import BytesIO
from pathlib import Path

# Simple CSV operations
import csv

In [3]:
pg = pgserver.get_server(
    "/tmp/hw3_pg",
    cleanup_mode="stop"
)

engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    version = conn.execute(text("SELECT version()")).scalar_one()
    print(version)

PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [4]:
DATA_URL = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
csv_path = Path("/content/hw3_bank_full.csv")

with urlopen(DATA_URL) as response:
    archive_data = response.read()

with ZipFile(BytesIO(archive_data)) as outer_zip:
    with ZipFile(BytesIO(outer_zip.read("bank.zip"))) as inner_zip:
        csv_data = inner_zip.read("bank-full.csv")

csv_path.write_bytes(csv_data)

print("CSV path:", csv_path)
print("File size:", csv_path.stat().st_size, "bytes")

CSV path: /content/hw3_bank_full.csv
File size: 4610348 bytes


In [5]:
with open(csv_path, encoding="utf-8", newline="") as file:
    reader = csv.reader(file, delimiter=";")

    columns = next(reader)
    first_row = next(reader)

print("Columns:", columns)
print("Number of columns:", len(columns))
print("First row:", first_row)

Columns: ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']
Number of columns: 17
First row: ['58', 'management', 'married', 'tertiary', 'no', '2143', 'yes', 'no', 'unknown', '5', 'may', '261', '1', '-1', '0', 'unknown', 'no']


Add a small helper function for SQL request execution

In [6]:
def run_query(query, params=None):
    with engine.connect() as conn:
        return pd.read_sql_query(
            text(query),
            conn,
            params=params
        )

Load the original CSV into the staging table using COPY FROM STDIN with a semicolon delimiter. All source columns are initially stored as TEXT to preserve their raw values before cleaning.

In [7]:
query = """
DROP TABLE IF EXISTS hw3_bank_clean CASCADE;
DROP TABLE IF EXISTS hw3_bank_staging CASCADE;

CREATE TABLE hw3_bank_staging (
    age_raw TEXT,
    job_raw TEXT,
    marital_raw TEXT,
    education_raw TEXT,
    default_raw TEXT,
    balance_raw TEXT,
    housing_raw TEXT,
    loan_raw TEXT,
    contact_raw TEXT,
    day_raw TEXT,
    month_raw TEXT,
    duration_raw TEXT,
    campaign_raw TEXT,
    pdays_raw TEXT,
    previous_raw TEXT,
    poutcome_raw TEXT,
    y_raw TEXT
);
"""

with engine.begin() as conn:
    conn.exec_driver_sql(query)

In [8]:
pd.read_sql("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_name = 'hw3_bank_staging'
    ORDER BY ordinal_position;
""", engine)

,column_name,data_type
0,age_raw,text
1,job_raw,text
2,marital_raw,text
3,education_raw,text
4,default_raw,text
5,balance_raw,text
6,housing_raw,text
7,loan_raw,text
8,contact_raw,text
9,day_raw,text


In [9]:
raw_conn = engine.raw_connection()

try:
  with raw_conn.cursor() as cur:
    with open(csv_path, 'r', encoding='utf-8', newline='') as file:
      cur.copy_expert(
          """
          COPY hw3_bank_staging(
            age_raw,
            job_raw,
            marital_raw,
            education_raw,
            default_raw,
            balance_raw,
            housing_raw,
            loan_raw,
            contact_raw,
            day_raw,
            month_raw,
            duration_raw,
            campaign_raw,
            pdays_raw,
            previous_raw,
            poutcome_raw,
            y_raw
          )
          FROM STDIN
          WITH(
            FORMAT csv,
            HEADER true,
            DELIMITER ';'
          )
          """,
          file
      )
  raw_conn.commit()
finally:
  raw_conn.close()

In [10]:
query = """
SELECT COUNT(*) AS loaded_rows
FROM hw3_bank_staging;
"""

pd.read_sql(query, engine)

,loaded_rows
0,45211


The staging table contains 45,211 records, matching the expected number of rows in bank-full.csv. No rows were removed during the import.

# Task 2. Staging and Typed DDL

### 2.1. Data Types and Cleaning Rules

The staging table stores all 17 source columns as `TEXT` to preserve raw values before conversion.

The clean table uses appropriate PostgreSQL data types based on the meaning of each column. The transformations are performed using `INSERT INTO ... SELECT`, mainly with `TRIM()`, `NULLIF()`, and explicit type casting.

| Source column(s) | Meaning | Clean type | Cleaning and validation |
|---|---|---|---|
| `age` | Client's age | `INTEGER` | Convert numeric text; CHECK: 1–120 |
| `job`, `education` | Employment and education categories | `TEXT` | Trim whitespace; convert empty strings and `unknown` to NULL |
| `marital` | Marital status | `TEXT` | Trim whitespace; convert empty strings to NULL |
| `default`, `housing`, `loan` | Credit default, housing loan, personal loan | `BOOLEAN` | Convert `yes/no` to TRUE/FALSE; empty strings to NULL |
| `balance` | Average yearly account balance in euros | `NUMERIC(12,2)` | Convert numeric text; negative balances are allowed |
| `contact` | Communication channel | `TEXT` | Preserve `unknown` as a category; empty strings become NULL |
| `day` | Day of the last contact | `SMALLINT` | Convert numeric text; CHECK: 1–31 |
| `month` | Month of the last contact | `TEXT` | Preserve month abbreviations; convert empty strings and `unknown` to NULL |
| `duration` | Last contact duration in seconds | `INTEGER` | Convert numeric text; CHECK: non-negative |
| `campaign` | Number of contacts during the current campaign | `INTEGER NOT NULL` | Convert numeric text; CHECK: at least 1 |
| `pdays` | Days since the previous campaign contact | `INTEGER` | Preserve `-1` as a special marker; CHECK: at least -1 |
| `previous` | Number of contacts before the current campaign | `INTEGER` | Convert numeric text; CHECK: non-negative |
| `poutcome` | Outcome of the previous marketing campaign | `TEXT` | Preserve `unknown` as a category; empty strings become NULL |
| `y` | Whether the client subscribed to a term deposit | `BOOLEAN NOT NULL` | Rename to `subscribed`; convert `yes/no` to TRUE/FALSE |

The clean table also includes `record_id`, a generated `BIGINT` primary key, because the source dataset does not provide a guaranteed unique record identifier.

Rows with missing `campaign` or `y` values are excluded during the transfer, as these fields are required by the clean schema.

Special values are handled according to their meaning. In particular, `unknown` is not automatically treated as NULL in every categorical column, and `pdays = -1` is preserved.

The dataset provides the day and month of contact, but not the year or exact timestamp. Therefore, these columns are kept separately rather than constructing an unreliable `DATE` or `TIMESTAMPTZ` value.

In [11]:
query = """
DROP TABLE IF EXISTS hw3_bank_clean CASCADE;

CREATE TABLE hw3_bank_clean (
    record_id BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    age INTEGER CHECK (age > 0 AND age <= 120),
    job TEXT,
    marital TEXT,
    education TEXT,
    has_default BOOLEAN,
    balance NUMERIC(12,2),
    housing BOOLEAN,
    loan BOOLEAN,
    contact TEXT,
    day SMALLINT CHECK (day BETWEEN 1 AND 31),
    month TEXT,
    duration INTEGER CHECK (duration >= 0),
    campaign INTEGER NOT NULL CHECK (campaign >= 1),
    pdays INTEGER CHECK (pdays >= -1),
    previous INTEGER CHECK (previous >= 0),
    poutcome TEXT,
    subscribed BOOLEAN NOT NULL
);
"""

with engine.begin() as conn:
    conn.exec_driver_sql(query)

In [12]:
display(run_query("""
    SELECT column_name, data_type, is_nullable
    FROM information_schema.columns
    WHERE table_schema = 'public'
      AND table_name = 'hw3_bank_clean'
    ORDER BY ordinal_position;
"""))

,column_name,data_type,is_nullable
0,record_id,bigint,NO
1,age,integer,YES
2,job,text,YES
3,marital,text,YES
4,education,text,YES
5,has_default,boolean,YES
6,balance,numeric,YES
7,housing,boolean,YES
8,loan,boolean,YES
9,contact,text,YES


In [13]:
query = """
SELECT
    NULLIF(TRIM(age_raw), '')::INTEGER AS age,
    NULLIF(NULLIF(TRIM(job_raw), ''), 'unknown') AS job,
    NULLIF(TRIM(marital_raw), '') AS marital,
    NULLIF(NULLIF(TRIM(education_raw), ''), 'unknown') AS education,
    NULLIF(TRIM(default_raw), '')::BOOLEAN AS has_default,
    NULLIF(TRIM(balance_raw), '')::NUMERIC(12,2) AS balance,
    NULLIF(TRIM(housing_raw), '')::BOOLEAN AS housing,
    NULLIF(TRIM(loan_raw), '')::BOOLEAN AS loan,
    NULLIF(TRIM(contact_raw), '') AS contact,
    NULLIF(TRIM(day_raw), '')::SMALLINT AS day,
    NULLIF(NULLIF(TRIM(month_raw), ''), 'unknown') AS month,
    NULLIF(TRIM(duration_raw), '')::INTEGER AS duration,
    TRIM(campaign_raw)::INTEGER AS campaign,
    NULLIF(TRIM(pdays_raw), '')::INTEGER AS pdays,
    NULLIF(TRIM(previous_raw), '')::INTEGER AS previous,
    NULLIF(TRIM(poutcome_raw), '') AS poutcome,
    TRIM(y_raw)::BOOLEAN AS subscribed
FROM hw3_bank_staging
WHERE NULLIF(TRIM(campaign_raw), '') IS NOT NULL
  AND NULLIF(TRIM(y_raw), '') IS NOT NULL
LIMIT 10;
"""

display(run_query(query))

,age,job,marital,education,has_default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,subscribed
0,58,management,married,tertiary,False,2143.0,True,False,unknown,5,may,261,1,-1,0,unknown,False
1,44,technician,single,secondary,False,29.0,True,False,unknown,5,may,151,1,-1,0,unknown,False
2,33,entrepreneur,married,secondary,False,2.0,True,True,unknown,5,may,76,1,-1,0,unknown,False
3,47,blue-collar,married,None,False,1506.0,True,False,unknown,5,may,92,1,-1,0,unknown,False
4,33,None,single,None,False,1.0,False,False,unknown,5,may,198,1,-1,0,unknown,False
5,35,management,married,tertiary,False,231.0,True,False,unknown,5,may,139,1,-1,0,unknown,False
6,28,management,single,tertiary,False,447.0,True,True,unknown,5,may,217,1,-1,0,unknown,False
7,42,entrepreneur,divorced,tertiary,True,2.0,True,False,unknown,5,may,380,1,-1,0,unknown,False
8,58,retired,married,primary,False,121.0,True,False,unknown,5,may,50,1,-1,0,unknown,False
9,43,technician,single,secondary,False,593.0,True,False,unknown,5,may,55,1,-1,0,unknown,False


In [14]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE NULLIF(TRIM(campaign_raw), '') IS NULL
    ) AS missing_campaign,
    COUNT(*) FILTER (
        WHERE NULLIF(TRIM(y_raw), '') IS NULL
    ) AS missing_target
FROM hw3_bank_staging;
"""

display(run_query(query))

,total_rows,missing_campaign,missing_target
0,45211,0,0


### 2.2. Filling Clean Table with Data

In [15]:
query = """
INSERT INTO hw3_bank_clean (
    age,
    job,
    marital,
    education,
    has_default,
    balance,
    housing,
    loan,
    contact,
    day,
    month,
    duration,
    campaign,
    pdays,
    previous,
    poutcome,
    subscribed
)
SELECT
    NULLIF(TRIM(age_raw), '')::INTEGER AS age,
    NULLIF(NULLIF(TRIM(job_raw), ''), 'unknown') AS job,
    NULLIF(TRIM(marital_raw), '') AS marital,
    NULLIF(NULLIF(TRIM(education_raw), ''), 'unknown') AS education,
    NULLIF(TRIM(default_raw), '')::BOOLEAN AS has_default,
    NULLIF(TRIM(balance_raw), '')::NUMERIC(12,2) AS balance,
    NULLIF(TRIM(housing_raw), '')::BOOLEAN AS housing,
    NULLIF(TRIM(loan_raw), '')::BOOLEAN AS loan,
    NULLIF(TRIM(contact_raw), '') AS contact,
    NULLIF(TRIM(day_raw), '')::SMALLINT AS day,
    NULLIF(NULLIF(TRIM(month_raw), ''), 'unknown') AS month,
    NULLIF(TRIM(duration_raw), '')::INTEGER AS duration,
    TRIM(campaign_raw)::INTEGER AS campaign,
    NULLIF(TRIM(pdays_raw), '')::INTEGER AS pdays,
    NULLIF(TRIM(previous_raw), '')::INTEGER AS previous,
    NULLIF(TRIM(poutcome_raw), '') AS poutcome,
    TRIM(y_raw)::BOOLEAN AS subscribed
FROM hw3_bank_staging
WHERE NULLIF(TRIM(campaign_raw), '') IS NOT NULL
  AND NULLIF(TRIM(y_raw), '') IS NOT NULL
"""

with engine.begin() as conn:
    conn.exec_driver_sql(
        "TRUNCATE TABLE hw3_bank_clean RESTART IDENTITY;"
    )
    conn.exec_driver_sql(query)

In [16]:
query = """
SELECT COUNT(*) AS loaded_rows
FROM hw3_bank_clean;
"""

pd.read_sql(query, engine)

,loaded_rows
0,45211


All 45,211 records were transferred from the staging table to the clean table. No rows were excluded because the required fields (`campaign` and `y`) contained no missing values. Raw values were converted to appropriate SQL types, with selected `unknown` markers normalized to NULL.

# Task 3. Data-quality audit

### 3.1. Row-count Verification

In [17]:
query = """
SELECT 'staging' AS layer, COUNT(*) AS n_rows
FROM hw3_bank_staging

UNION ALL

SELECT 'clean' AS layer, COUNT(*) AS n_rows
FROM hw3_bank_clean;
"""

display(run_query(query))

,layer,n_rows
0,staging,45211
1,clean,45211


The staging and clean tables contain the same number of records (45,211). No rows were excluded during cleaning, since the required fields contained no missing values.

### 3.2. Duplicate Check

In [18]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (
      age,
      job,
      marital,
      education,
      has_default,
      balance,
      housing,
      loan,
      contact,
      day,
      month,
      duration,
      campaign,
      pdays,
      previous,
      poutcome,
      subscribed
    )) AS distinct_attribute_combinations,
    COUNT(*) - COUNT(DISTINCT (
      age,
      job,
      marital,
      education,
      has_default,
      balance,
      housing,
      loan,
      contact,
      day,
      month,
      duration,
      campaign,
      pdays,
      previous,
      poutcome,
      subscribed
    )) AS duplicate_rows
FROM hw3_bank_clean;
"""

display(run_query(query))

,total_rows,distinct_attribute_combinations,duplicate_rows
0,45211,45211,0


No exact duplicate records were found across the 17 business attributes in the clean table. However, the dataset does not contain a unique customer or contact identifier, so potential repeated contacts cannot be reliably identified.

### 3.3. NULL and Sentinel Values Audit

In [19]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE job IS NULL) AS missing_job,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE job IS NULL)
        / NULLIF(COUNT(*), 0),
        2
    ) AS pct_missing_job,
    COUNT(*) FILTER (WHERE education IS NULL) AS missing_education,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE education IS NULL)
        / NULLIF(COUNT(*), 0),
        2
    ) AS pct_missing_education,
    COUNT(*) FILTER (WHERE contact IS NULL) AS missing_contact,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE contact IS NULL)
        / NULLIF(COUNT(*), 0),
        2
    ) AS pct_missing_contact,
    COUNT(*) FILTER (WHERE poutcome IS NULL) AS missing_poutcome,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE poutcome IS NULL)
        / NULLIF(COUNT(*), 0),
        2
    ) AS pct_missing_poutcome,
    COUNT(*) FILTER (WHERE balance IS NULL) AS missing_balance,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE balance IS NULL)
        / NULLIF(COUNT(*), 0),
        2
    ) AS pct_missing_balance,
    COUNT(*) FILTER (WHERE age IS NULL) AS missing_age,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE age IS NULL)
        / NULLIF(COUNT(*), 0),
        2
    ) AS pct_missing_age
FROM hw3_bank_clean;
"""

display(run_query(query))

,total_rows,missing_job,pct_missing_job,missing_education,pct_missing_education,missing_contact,pct_missing_contact,missing_poutcome,pct_missing_poutcome,missing_balance,pct_missing_balance,missing_age,pct_missing_age
0,45211,288,0.64,1857,4.11,0,0.0,0,0.0,0,0.0,0,0.0


In [20]:
query = """
SELECT
    COUNT(*) FILTER (
        WHERE job_raw = 'unknown'
    ) AS unknown_job,
    COUNT(*) FILTER (
        WHERE education_raw = 'unknown'
    ) AS unknown_education,
    COUNT(*) FILTER (
        WHERE contact_raw = 'unknown'
    ) AS unknown_contact,
    COUNT(*) FILTER (
        WHERE poutcome_raw = 'unknown'
    ) AS unknown_poutcome,
    COUNT(*) FILTER (
        WHERE pdays_raw = '-1'
    ) AS never_contacted_before
FROM hw3_bank_staging;
"""

display(run_query(query))

,unknown_job,unknown_education,unknown_contact,unknown_poutcome,never_contacted_before
0,288,1857,13020,36959,36954


In [21]:

query = """
SELECT
    field_name,
    COUNT(*) FILTER (
        WHERE raw_value IS NULL
    ) AS raw_nulls,
    COUNT(*) FILTER (
        WHERE raw_value IS NOT NULL AND TRIM(raw_value) = ''
    ) AS empty_strings,
    COUNT(*) FILTER (
        WHERE TRIM(raw_value) = '?'
    ) AS question_marks,
    COUNT(*) FILTER (
        WHERE LOWER(TRIM(raw_value)) = 'unknown'
    ) AS unknown_markers
FROM hw3_bank_staging AS s
CROSS JOIN LATERAL (
    VALUES
        ('age', s.age_raw),
        ('job', s.job_raw),
        ('marital', s.marital_raw),
        ('education', s.education_raw),
        ('default', s.default_raw),
        ('balance', s.balance_raw),
        ('housing', s.housing_raw),
        ('loan', s.loan_raw),
        ('contact', s.contact_raw),
        ('day', s.day_raw),
        ('month', s.month_raw),
        ('duration', s.duration_raw),
        ('campaign', s.campaign_raw),
        ('pdays', s.pdays_raw),
        ('previous', s.previous_raw),
        ('poutcome', s.poutcome_raw),
        ('y', s.y_raw)
) AS fields(field_name, raw_value)
GROUP BY field_name
ORDER BY field_name;
"""

display(run_query(query))

,field_name,raw_nulls,empty_strings,question_marks,unknown_markers
0,age,0,0,0,0
1,balance,0,0,0,0
2,campaign,0,0,0,0
3,contact,0,0,0,13020
4,day,0,0,0,0
5,default,0,0,0,0
6,duration,0,0,0,0
7,education,0,0,0,1857
8,housing,0,0,0,0
9,job,0,0,0,288


The NULL audit was performed on six columns of the clean table. Missing values were found in `job` (288 records, 0.64%) and `education` (1,857 records, 4.11%). The other checked columns (`contact`, `poutcome`, `balance`, and `age`) contained no SQL NULL values.

The missing values in `job` and `education` match the counts of `unknown` markers in the staging table. These markers were converted to NULL during cleaning.

However, not all `unknown` markers were converted. In particular, `contact` contains 13,020 `unknown` values (28.80%), while `poutcome` contains 36,959 (81.75%). They were preserved as separate categories. This shows that a zero NULL rate does not necessarily mean that all values are known or informative.

Another special value is `pdays = -1`, found in 36,954 records (81.74%). It indicates that the client was not previously contacted in an earlier campaign. This value was preserved because it has a defined meaning and is not an invalid negative number.

Overall, missing or unspecified information is mainly concentrated in categorical attributes and previous campaign history. These values should be handled carefully during further analysis and ML preprocessing rather than removed automatically.

The raw-value audit across all 17 source columns found no SQL NULL values, empty strings, or `?` markers. The `unknown` marker appeared only in `job`, `education`, `contact`, and `poutcome`. This confirms that missing or unspecified categorical information in the source dataset is mainly represented by explicit `unknown` values rather than empty fields.

# Task 4. Distributions and Anomalies

### 4.1. Categorical Distributions

In [22]:
query = """
SELECT
    subscribed,
    COUNT(*) AS n_rows,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS pct_rows
FROM hw3_bank_clean
GROUP BY subscribed
ORDER BY n_rows DESC, subscribed;
"""

display(run_query(query))

,subscribed,n_rows,pct_rows
0,False,39922,88.3
1,True,5289,11.7


The target variable is imbalanced: 39,922 clients (88.30%) did not subscribe to a term deposit, while 5,289 (11.70%) subscribed.

This class imbalance should be considered in future ML classification tasks. Accuracy alone may be misleading, so metrics such as precision, recall, F1-score, and PR-AUC would be useful for model evaluation.

In [23]:
query = """
SELECT
    COALESCE(job, 'Missing') AS job_category,
    COUNT(*) AS n_rows,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS pct_rows
FROM hw3_bank_clean
GROUP BY job_category
ORDER BY n_rows DESC, job_category;
"""

display(run_query(query))

,job_category,n_rows,pct_rows
0,blue-collar,9732,21.53
1,management,9458,20.92
2,technician,7597,16.80
3,admin.,5171,11.44
4,services,4154,9.19
5,retired,2264,5.01
6,self-employed,1579,3.49
7,entrepreneur,1487,3.29
8,unemployed,1303,2.88
9,housemaid,1240,2.74


In [24]:
query = """
SELECT
    COALESCE(education, 'Missing') AS education_category,
    COUNT(*) AS n_rows,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS pct_rows
FROM hw3_bank_clean
GROUP BY education_category
ORDER BY n_rows DESC, education_category;
"""

display(run_query(query))

,education_category,n_rows,pct_rows
0,secondary,23202,51.32
1,tertiary,13301,29.42
2,primary,6851,15.15
3,Missing,1857,4.11


In [25]:
query = """
SELECT
    COALESCE(contact, 'Missing') AS contact_category,
    COUNT(*) AS n_rows,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS pct_rows
FROM hw3_bank_clean
GROUP BY contact_category
ORDER BY n_rows DESC, contact_category;
"""

display(run_query(query))

,contact_category,n_rows,pct_rows
0,cellular,29285,64.77
1,unknown,13020,28.80
2,telephone,2906,6.43


The most common job categories are `blue-collar` (21.53%), `management` (20.92%), and `technician` (16.80%). Together, they represent 59.25% of all records. The `job` field also contains 288 missing values (0.64%).

For education, `secondary` is the most common category (51.32%), followed by `tertiary` (29.42%) and `primary` (15.15%). Another 4.11% of records have missing education information.

Most contacts were made via `cellular` (64.77%), while `telephone` accounts for only 6.43%. However, the contact method is recorded as `unknown` in 28.80% of cases, which may limit further analysis of communication channels.

These distributions show that several categories dominate the dataset. Missing and unknown values should also be considered during feature preparation for ML.

### 4.2. Numeric Summary

In [26]:
query = """
SELECT
    v.feature,
    COUNT(v.value) AS n_values,
    MIN(v.value) AS min_value,
    MAX(v.value) AS max_value,
    ROUND(AVG(v.value), 2) AS avg_value,
    ROUND(STDDEV_SAMP(v.value), 2) AS std_value
FROM hw3_bank_clean AS b
CROSS JOIN LATERAL (
    VALUES
        (1, 'age', b.age::NUMERIC),
        (2, 'balance', b.balance),
        (3, 'duration', b.duration::NUMERIC),
        (4, 'campaign', b.campaign::NUMERIC),
        (5, 'previous', b.previous::NUMERIC),
        (6, 'pdays (all records)', b.pdays::NUMERIC),
        -- Exclude the -1 sentinel from this calculation
        (7, 'pdays (previously contacted)',
            NULLIF(b.pdays, -1)::NUMERIC)
) AS v(sort_order, feature, value)
GROUP BY v.sort_order, v.feature
ORDER BY v.sort_order;
"""

display(run_query(query))


,feature,n_values,min_value,max_value,avg_value,std_value
0,age,45211,18.0,95.0,40.94,10.62
1,balance,45211,-8019.0,102127.0,1362.27,3044.77
2,duration,45211,0.0,4918.0,258.16,257.53
3,campaign,45211,1.0,63.0,2.76,3.10
4,previous,45211,0.0,275.0,0.58,2.30
5,pdays (all records),45211,-1.0,871.0,40.20,100.13
6,pdays (previously contacted),8257,1.0,871.0,224.58,115.34


All six numeric features contain 45,211 non-NULL values.

The average client age is 40.94 years, with values ranging from 18 to 95.

The average account balance is €1,362.27, but values range from -€8,019 to €102,127. The high standard deviation (€3,044.77) suggests considerable variability and possible extreme values. Negative balances are valid and should not automatically be treated as errors.

The average call duration is 258.16 seconds, with values ranging from 0 to 4,918 seconds. The maximum duration is unusually high and may require further investigation.

The average number of contacts during the current campaign is 2.76, while the maximum is 63. Previous contacts range from 0 to 275, with an average of only 0.58. These large maximum values may indicate outliers.

The `pdays` variable requires special handling because `-1` means that the client was not previously contacted. When all records are included, the average is 40.20 days. However, among the 8,257 records with `pdays >= 0`, the average is 224.58 days. This shows that sentinel values can make numerical statistics misleading.

Overall, several numeric features have wide ranges and potentially extreme values. These should be investigated using domain-specific checks before deciding whether any values need to be changed or excluded.

### 4.3. Anomaly Checks

In [27]:
query = """
SELECT
    f.check_name,
    COUNT(*) FILTER (
        WHERE f.is_flagged
    ) AS n_flagged,
    ROUND(
        COUNT(*) FILTER (WHERE f.is_flagged) * 100.0
        / NULLIF(COUNT(*), 0),
        2
    ) AS pct_rows
FROM hw3_bank_clean AS b
CROSS JOIN LATERAL (
    VALUES
        -- Financial values
        (1, 'Negative balance', b.balance < 0),
        (2, 'Balance > 10000 EUR', b.balance > 10000),

        -- Call duration
        (3, 'Zero call duration', b.duration = 0),
        (4, 'Call duration > 3600 sec', b.duration > 3600),

        -- Number of contacts
        (5, 'Campaign contacts > 20', b.campaign > 20),
        (6, 'Previous contacts > 50', b.previous > 50),

        -- Time since previous contact
        (7, 'Pdays > 365', b.pdays > 365)
) AS f(sort_order, check_name, is_flagged)
GROUP BY f.sort_order, f.check_name
ORDER BY f.sort_order;
"""

anomaly_df = run_query(query)
display(anomaly_df)

,check_name,n_flagged,pct_rows
0,Negative balance,3766,8.33
1,Balance > 10000 EUR,829,1.83
2,Zero call duration,3,0.01
3,Call duration > 3600 sec,3,0.01
4,Campaign contacts > 20,244,0.54
5,Previous contacts > 50,4,0.01
6,Pdays > 365,643,1.42


In [28]:
query = """
SELECT
    record_id,
    age,
    balance,
    duration,
    campaign,
    previous,
    pdays,
    subscribed
FROM hw3_bank_clean
WHERE
    duration > 3600
    OR campaign > 20
    OR previous > 50
ORDER BY duration DESC, campaign DESC
LIMIT 10;
"""

display(run_query(query))

,record_id,age,balance,duration,campaign,previous,pdays,subscribed
0,24149,59,6573.0,4918,1,0,-1,False
1,9948,59,1321.0,3881,3,0,-1,True
2,44603,45,955.0,3785,1,0,-1,False
3,16638,38,593.0,1484,24,0,-1,True
4,4016,34,305.0,1245,25,0,-1,False
5,5074,35,432.0,1094,55,0,-1,False
6,17955,31,0.0,1037,24,0,-1,False
7,17874,46,271.0,1013,29,0,-1,True
8,10467,27,806.0,768,21,0,-1,False
9,3332,50,461.0,651,32,0,-1,True


The anomaly audit identified several potentially unusual values across the 45,211 records.

**Account balance:** 3,766 records (8.33%) have negative balances, and 829 (1.83%) have balances above €10,000. These values are not necessarily errors, as negative or unusually high account balances can be legitimate.

**Call duration:** Only 3 records (0.01%) have zero-duration calls, and another 3 (0.01%) have calls longer than one hour. The longest call lasted 4,918 seconds (approximately 82 minutes). These cases may require further investigation.

**Contact frequency:** 244 records (0.54%) have more than 20 contacts during the current campaign, while only 4 (0.01%) have more than 50 previous contacts. Such extreme values are rare but may affect statistical analysis and ML models.

**Previous contact interval:** 643 records (1.42%) have `pdays > 365`. This represents approximately 7.79% of the 8,257 records with a known previous contact interval. The special value `pdays = -1` is not considered an anomaly because it represents the absence of a previous contact.

**Handling decision:** All flagged values will be retained in the clean table. The selected thresholds are exploratory and do not prove that the records contain errors. Potential outliers should be investigated or flagged during future ML preprocessing rather than automatically corrected or removed.

Overall, most flagged cases involve negative or high account balances, while extreme call durations and contact counts are relatively rare.

# Task 5. Business Questions and DQL Queries

### Query 1. Subscription Rate by Job

**Business Question:** How does the subscription rate vary by job category?

In [29]:
query = """
SELECT
    COALESCE(job, 'Missing') AS job_category,
    COUNT(*) AS total_records,
    COUNT(*) FILTER (WHERE subscribed = TRUE) AS subscribed_records,
    ROUND(
        COUNT(*) FILTER (WHERE subscribed = TRUE) * 100.0 / COUNT(*),
        2
    ) AS subscription_rate_pct
FROM hw3_bank_clean
GROUP BY job_category
ORDER BY subscription_rate_pct DESC;
"""

display(run_query(query))

,job_category,total_records,subscribed_records,subscription_rate_pct
0,student,938,269,28.68
1,retired,2264,516,22.79
2,unemployed,1303,202,15.50
3,management,9458,1301,13.76
4,admin.,5171,631,12.20
5,self-employed,1579,187,11.84
6,Missing,288,34,11.81
7,technician,7597,840,11.06
8,services,4154,369,8.88
9,housemaid,1240,109,8.79


**Interpretation:**

Subscription rates vary considerably across job categories. Students have the highest subscription rate (28.68%), followed by retired clients (22.79%), while blue-collar workers have the lowest rate (7.27%). However, management accounts for the largest number of successful subscriptions (1,301), showing that a high conversion rate does not necessarily mean the largest number of subscriptions.

### Query 2. Subscription Rate by Contact Method

**Business Question:** How does the subscription rate vary by contact method?

In [30]:
query = """
SELECT
    contact,
    COUNT(*) AS total_records,
    COUNT(*) FILTER (WHERE subscribed = TRUE) AS subscribed_records,
    ROUND(
        COUNT(*) FILTER (WHERE subscribed = TRUE) * 100.0 / COUNT(*),
        2
    ) AS subscription_rate_pct
FROM hw3_bank_clean
GROUP BY contact
ORDER BY subscription_rate_pct DESC;
"""

display(run_query(query))

,contact,total_records,subscribed_records,subscription_rate_pct
0,cellular,29285,4369,14.92
1,telephone,2906,390,13.42
2,unknown,13020,530,4.07


**Interpretation:**

Cellular contacts have the highest subscription rate (14.92%), followed by telephone contacts (13.42%). Records with an unknown contact method have a much lower subscription rate (4.07%). This suggests an association between contact method and subscription outcome, but it does not prove that the communication channel itself caused the difference.

### Query 3. Subscription Rate by Campaign Contact Number

**Business Question:** How does the number of campaign contacts relate to subscription rates?

In [31]:
query = """
WITH campaign_groups AS (
    SELECT
        CASE
            WHEN campaign = 1 THEN '1 contact'
            WHEN campaign = 2 THEN '2 contacts'
            WHEN campaign BETWEEN 3 AND 4 THEN '3 to 4 contacts'
            WHEN campaign BETWEEN 5 AND 8 THEN '5 to 8 contacts'
            ELSE '9+ contacts'
        END AS contact_group,
        CASE
            WHEN campaign = 1 THEN 1
            WHEN campaign = 2 THEN 2
            WHEN campaign BETWEEN 3 AND 4 THEN 3
            WHEN campaign BETWEEN 5 AND 8 THEN 4
            ELSE 5
        END AS group_order,
        subscribed
    FROM hw3_bank_clean
)
SELECT
    contact_group,
    COUNT(*) AS total_records,
    COUNT(*) FILTER (
        WHERE subscribed = TRUE
    ) AS subscribed_records,
    ROUND(
        COUNT(*) FILTER (WHERE subscribed = TRUE) * 100.0
        / COUNT(*),
        2
    ) AS subscription_rate_pct
FROM campaign_groups
GROUP BY group_order, contact_group
ORDER BY group_order;
"""

display(run_query(query))

,contact_group,total_records,subscribed_records,subscription_rate_pct
0,1 contact,17544,2561,14.60
1,2 contacts,12505,1401,11.20
2,3 to 4 contacts,9043,935,10.34
3,5 to 8 contacts,4330,310,7.16
4,9+ contacts,1789,82,4.58


**Interpretation:**

Subscription rates decrease as the number of campaign contacts increases. Records with only one contact have the highest subscription rate (14.60%), followed by two contacts (11.20%), three to four contacts (10.34%), five to eight contacts (7.16%), and nine or more contacts (4.58%).

This pattern suggests an association between contact frequency and subscription outcomes. However, it does not prove that additional contacts reduce the chance of success, since less responsive clients may require more follow-up contacts.

### Query 4. Subscription Rate by Previous Campaign Outcome

**Business Question:** How does the previous campaign outcome relate to subscription success?

In [32]:
query = """
SELECT
    poutcome,
    COUNT(*) AS total_records,
    COUNT(*) FILTER (WHERE subscribed = TRUE) AS subscribed_records,
    ROUND(
        COUNT(*) FILTER (WHERE subscribed = TRUE) * 100.0 / COUNT(*),
        2
    ) AS subscription_rate_pct
FROM hw3_bank_clean
GROUP BY poutcome
ORDER BY subscription_rate_pct DESC;
"""

display(run_query(query))

,poutcome,total_records,subscribed_records,subscription_rate_pct
0,success,1511,978,64.73
1,other,1840,307,16.68
2,failure,4901,618,12.61
3,unknown,36959,3386,9.16


**Interpretation:**

Records with a successful previous campaign outcome have a much higher subscription rate (64.73%) than those with previous failures (12.61%) or unknown outcomes (9.16%). The `other` category has a subscription rate of 16.68%. Previous campaign success appears strongly associated with the current outcome, although the large `unknown` category limits interpretation.

### Query 5. Subscription Rate by Marital Status

**Business Question:** How does the subscription rate vary by marital status?

In [33]:
query = """
SELECT
    marital,
    COUNT(*) AS total_records,
    COUNT(*) FILTER (WHERE subscribed = TRUE) AS subscribed_records,
    ROUND(
        COUNT(*) FILTER (WHERE subscribed = TRUE) * 100.0 / COUNT(*),
        2
    ) AS subscription_rate_pct
FROM hw3_bank_clean
GROUP BY marital
ORDER BY subscription_rate_pct DESC;
"""

display(run_query(query))

,marital,total_records,subscribed_records,subscription_rate_pct
0,single,12790,1912,14.95
1,divorced,5207,622,11.95
2,married,27214,2755,10.12


**Interpretation:**

Single clients have the highest subscription rate (14.95%), followed by divorced clients (11.95%) and married clients (10.12%). Married clients account for the largest number of successful subscriptions (2,755) because they form the largest group in the dataset. These differences indicate an association between marital status and subscription outcome, but other factors, such as age and occupation, may also contribute.

### Query 6. Housing Loan Rate by Marital Status

**Business Question:** How does having a housing loan vary by marital status?

In [34]:
query = """
SELECT
    marital,
    COUNT(*) AS total_records,
    COUNT(*) FILTER (WHERE housing = TRUE) AS clients_housing_loan,
    ROUND(
        COUNT(*) FILTER (WHERE housing = TRUE) * 100.0 / COUNT(*),
        2
    ) AS housing_loan_rate_pct
FROM hw3_bank_clean
GROUP BY marital
ORDER BY housing_loan_rate_pct DESC;
"""

display(run_query(query))

,marital,total_records,clients_housing_loan,housing_loan_rate_pct
0,married,27214,15321,56.30
1,divorced,5207,2907,55.83
2,single,12790,6902,53.96


**Interpretation:**

The share of records with a housing loan is highest among married clients (56.30%), followed by divorced (55.83%) and single clients (53.96%). The differences are relatively small, with only 2.34 percentage points between the highest and lowest groups. This suggests that housing loan prevalence is fairly similar across marital status categories in this dataset.

### Query 7. Marital Status Distribution by Age Group

Business Question: How does marital status vary across different age groups?

In [35]:
query = """
WITH age_groups AS (
    SELECT
        CASE
            WHEN age < 25 THEN '18-24'
            WHEN age < 35 THEN '25-34'
            WHEN age < 45 THEN '35-44'
            WHEN age < 55 THEN '45-54'
            WHEN age < 65 THEN '55-64'
            ELSE '65+'
        END AS age_group,
        marital
    FROM hw3_bank_clean
    WHERE age IS NOT NULL
      AND marital IS NOT NULL
)
SELECT
    age_group,
    marital,
    COUNT(*) AS n_records,
    ROUND(
        COUNT(*) * 100.0
        / SUM(COUNT(*)) OVER (PARTITION BY age_group),
        2
    ) AS pct_within_age_group
FROM age_groups
GROUP BY age_group, marital
ORDER BY age_group, n_records DESC;
"""

display(run_query(query))

,age_group,marital,n_records,pct_within_age_group
0,18-24,single,711,87.89
1,18-24,married,97,11.99
2,18-24,divorced,1,0.12
3,25-34,single,7442,52.39
4,25-34,married,6058,42.65
5,25-34,divorced,704,4.96
6,35-44,married,9322,64.14
7,35-44,single,3431,23.61
8,35-44,divorced,1781,12.25
9,45-54,married,7355,73.86


**Interpretation:**

Marital status distributions vary considerably across age groups. Single clients represent 87.89% of records aged 18-24 and 52.39% of those aged 25-34, while married clients become the largest group from age 35 onwards.

The share of married clients increases from 64.14% in the 35-44 age group to 77.16% among clients aged 65 and older. Divorced clients also account for a larger share in older age groups, reaching 19.51% among those aged 65 and older.

### Query 8. Top Job Categories by Education Level

Business Question: What are the five most common job categories within each education level?

In [36]:
query = """
WITH job_counts AS (
    SELECT
        COALESCE(education, 'Missing') AS education_level,
        COALESCE(job, 'Missing') AS job_category,
        COUNT(*) AS n_records
    FROM hw3_bank_clean
    GROUP BY
        COALESCE(education, 'Missing'),
        COALESCE(job, 'Missing')
),
ranked_jobs AS (
    SELECT
        education_level,
        job_category,
        n_records,
        ROUND(
            n_records * 100.0
            / SUM(n_records) OVER (PARTITION BY education_level),
            2
        ) AS pct_within_education,
        ROW_NUMBER() OVER (
            PARTITION BY education_level
            ORDER BY n_records DESC, job_category
        ) AS job_rank
    FROM job_counts
)
SELECT
    education_level,
    job_category,
    n_records,
    pct_within_education
FROM ranked_jobs
WHERE job_rank <= 5
ORDER BY education_level, job_rank;
"""

display(run_query(query))

,education_level,job_category,n_records,pct_within_education
0,Missing,blue-collar,454,24.45
1,Missing,management,242,13.03
2,Missing,technician,242,13.03
3,Missing,admin.,171,9.21
4,Missing,student,163,8.78
5,primary,blue-collar,3758,54.85
6,primary,retired,795,11.60
7,primary,housemaid,627,9.15
8,primary,services,345,5.04
9,primary,management,294,4.29


**Interpretation:**

Job distributions differ substantially by education level. Blue-collar workers represent 54.85% of records with primary education, while management is the dominant category among clients with tertiary education (58.65%).

For secondary education, the largest categories are blue-collar (23.15%) and technician (22.54%), followed by administrative workers (18.18%). These differences show a strong association between education level and occupation in the dataset.

### Query 9. Unknown and Missing Values in Categorical Fields

Business Question: Which categorical fields contain unknown or missing values, and how frequently do they occur?

In [37]:
query = """
SELECT
    v.field_name,
    COUNT(*) FILTER (
        WHERE v.field_value IS NULL
    ) AS null_values,
    COUNT(*) FILTER (
        WHERE v.field_value ILIKE 'unknown'
    ) AS unknown_values,
    COUNT(*) FILTER (
        WHERE v.field_value IS NOT NULL
          AND v.field_value IS DISTINCT FROM 'unknown'
    ) AS known_values,
    COUNT(DISTINCT v.field_value) AS distinct_non_null_values,
    ROUND(
        COUNT(*) FILTER (
            WHERE v.field_value IS NULL
               OR v.field_value ILIKE 'unknown'
        ) * 100.0 / NULLIF(COUNT(*), 0),
        2
    ) AS pct_missing_or_unknown
FROM hw3_bank_clean AS b
CROSS JOIN LATERAL (
    VALUES
        ('job', b.job),
        ('marital', b.marital),
        ('education', b.education),
        ('contact', b.contact),
        ('month', b.month),
        ('poutcome', b.poutcome)
) AS v(field_name, field_value)
GROUP BY v.field_name
ORDER BY pct_missing_or_unknown DESC, v.field_name;
"""

display(run_query(query))

,field_name,null_values,unknown_values,known_values,distinct_non_null_values,pct_missing_or_unknown
0,poutcome,0,36959,8252,4,81.75
1,contact,0,13020,32191,3,28.80
2,education,1857,0,43354,3,4.11
3,job,288,0,44923,11,0.64
4,marital,0,0,45211,3,0.00
5,month,0,0,45211,12,0.00


**Interpretation:**

Missing or unknown values are concentrated in a few categorical fields. The highest proportion is found in `poutcome` (81.75%), followed by `contact` (28.80%), while `education` and `job` contain 4.11% and 0.64% missing values, respectively.

No missing or unknown values were found in `marital` or `month`. The results also reflect the cleaning rules: `unknown` was converted to NULL in `job` and `education`, but preserved as a category in `contact` and `poutcome`.

### Query 10. Subscription Rate by Loan Combination

Business Question: How do housing loans and personal loans, individually or together, relate to subscription outcomes?

In [38]:
query = """
SELECT
    housing AS housing_loan,
    loan AS personal_loan,
    COUNT(*) AS total_records,
    COUNT(*) FILTER (
        WHERE subscribed = TRUE
    ) AS subscribed_records,
    ROUND(
        COUNT(*) FILTER (WHERE subscribed = TRUE) * 100.0
        / COUNT(*),
        2
    ) AS subscription_rate_pct
FROM hw3_bank_clean
WHERE housing IS NOT NULL
  AND loan IS NOT NULL
GROUP BY housing, loan
ORDER BY housing DESC, loan DESC;
"""

display(run_query(query))

,housing_loan,personal_loan,total_records,subscribed_records,subscription_rate_pct
0,True,True,4367,265,6.07
1,True,False,20763,1670,8.04
2,False,True,2877,219,7.61
3,False,False,17204,3135,18.22


**Interpretation:**

Records without either a housing loan or a personal loan have the highest subscription rate (18.22%), with 3,135 successful subscriptions out of 17,204 records. The lowest subscription rate (6.07%) is observed among clients with both types of loans.

Records with only a housing loan have a subscription rate of 8.04%, compared with 7.61% for those with only a personal loan. These results suggest an association between loan status and subscription outcomes, but they do not establish a causal relationship.

### Query 11. Job Categories in a Selected Customer Segment

Business Question: Which job categories are most common among clients aged 25-40 with secondary or tertiary education who have neither a housing loan nor a personal loan?

In [39]:
query = """
SELECT
    COALESCE(job, 'Missing') AS job_category,
    COUNT(*) AS n_records,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS pct_of_segment
FROM hw3_bank_clean
WHERE age BETWEEN 25 AND 40
  AND education IN ('secondary', 'tertiary')
  AND housing IS NOT NULL
  AND loan IS NOT NULL
  AND NOT (housing OR loan)
GROUP BY COALESCE(job, 'Missing')
ORDER BY n_records DESC, job_category
LIMIT 10;
"""

display(run_query(query))

,job_category,n_records,pct_of_segment
0,management,2188,31.19
1,technician,1665,23.73
2,admin.,806,11.49
3,blue-collar,589,8.40
4,services,522,7.44
5,self-employed,346,4.93
6,student,318,4.53
7,unemployed,273,3.89
8,entrepreneur,159,2.27
9,housemaid,101,1.44


**Interpretation:**

Among records aged 25–40 with secondary or tertiary education and no housing or personal loans, management is the most common job category (31.19%), followed by technician (23.73%) and administrative workers (11.49%).

Together, these three categories represent 66.41% of the selected segment. The results show that management and technical occupations are particularly common within this customer group. Only the ten most frequent job categories are displayed.

### Query 12. Call Duration and Subscription Outcome

Business Question: How does call duration relate to subscription outcomes, and why is duration unavailable for prediction before the call ends?

In [40]:
query = """
WITH duration_groups AS (
    SELECT
        CASE
            WHEN duration = 0 THEN '0 seconds'
            WHEN duration <= 60 THEN '1-60 seconds'
            WHEN duration <= 180 THEN '61-180 seconds'
            WHEN duration <= 300 THEN '181-300 seconds'
            WHEN duration <= 600 THEN '301-600 seconds'
            ELSE 'Over 600 seconds'
        END AS duration_group,
        CASE
            WHEN duration = 0 THEN 1
            WHEN duration <= 60 THEN 2
            WHEN duration <= 180 THEN 3
            WHEN duration <= 300 THEN 4
            WHEN duration <= 600 THEN 5
            ELSE 6
        END AS group_order,
        subscribed
    FROM hw3_bank_clean
    WHERE duration IS NOT NULL
)
SELECT
    duration_group,
    COUNT(*) AS total_records,
    COUNT(*) FILTER (
        WHERE subscribed = TRUE
    ) AS subscribed_records,
    ROUND(
        COUNT(*) FILTER (WHERE subscribed = TRUE) * 100.0
        / COUNT(*),
        2
    ) AS subscription_rate_pct
FROM duration_groups
GROUP BY group_order, duration_group
ORDER BY group_order;
"""

display(run_query(query))

,duration_group,total_records,subscribed_records,subscription_rate_pct
0,0 seconds,3,0,0.00
1,1-60 seconds,4763,9,0.19
2,61-180 seconds,17894,700,3.91
3,181-300 seconds,10277,1122,10.92
4,301-600 seconds,8484,1625,19.15
5,Over 600 seconds,3790,1833,48.36


**Interpretation:**

Subscription rates increase substantially with call duration. Calls lasting 1-60 seconds have a subscription rate of only 0.19%, compared with 10.92% for calls lasting 181-300 seconds and 48.36% for calls longer than 600 seconds.

This shows a strong association, but does not prove that longer calls cause successful subscriptions. For a model predicting subscription before a call, `duration` must be excluded because the final call length is unknown at prediction time and would introduce data leakage.

# Task 6. Reflection

In this homework, I loaded the UCI Bank Marketing dataset into PostgreSQL and created a clean table for SQL analysis. The dataset contains 45,211 records. No rows were lost during cleaning, and I found no exact duplicates across the source columns.

The main data quality issue was the `unknown` value. It appears in several categorical columns, especially `poutcome` (81.75%) and `contact` (28.80%). I converted `unknown` to NULL for `job` and `education`, but kept it for `contact` and `poutcome`. I also kept `pdays = -1`, because it means that there was no previous contact. I did not remove extreme balances or long calls without evidence that they were incorrect.

The SQL analysis showed that only 11.70% of records have a positive subscription result. Subscription rates also differ by job, previous campaign outcome and loan status. These results show associations, not necessarily causal relationships.

For a classification model that predicts subscription before a call, I would use customer details and information available from earlier campaigns. I would exclude `duration`, as the final call length is unknown before the call ends. I would also check when `campaign` is measured. For future improvements, I would add automatic checks for data types, missing values and unusual values, and use evaluation metrics suitable for imbalanced classes.